# Multi-Seed Rerun — With Google Drive Saving

**Every model is saved to Google Drive immediately after training.**  
If Colab disconnects, your finished models are safe. Just reconnect and resume.

---
**Before running:** Set your seed below.

In [ ]:
# ── SET SEED BEFORE RUNNING ─────────────────────────────────────
CURRENT_SEED = 123   # Change to 456 for second rerun
# ───────────────────────────────────────────────────────────────
print(f'Seed: {CURRENT_SEED}')

## Step 1 — Mount Google Drive
This saves all checkpoints and results directly to your Drive.  
Even if Colab disconnects, everything is safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/skin_lesion_research/'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Drive mounted. Saving to: {SAVE_DIR}')

In [ ]:
!pip install -q kaggle timm scikit-learn matplotlib seaborn pandas Pillow opencv-python-headless

In [ ]:
import os, io, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFilter
from copy import deepcopy

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

SEED = CURRENT_SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25
NUM_CLASSES = 7

CLASS_NAMES = {
    'nv': 'Melanocytic Nevi', 'mel': 'Melanoma',
    'bkl': 'Benign Keratosis', 'bcc': 'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratoses', 'vasc': 'Vascular Lesions', 'df': 'Dermatofibroma'
}
CLASS_TO_IDX = {k: i for i, k in enumerate(CLASS_NAMES.keys())}

DEGRADATION_PARAMS = {
    'Gaussian Noise':          [0, 10, 25, 50, 75, 100],
    'Gaussian Blur':           [0, 1, 2, 3, 5, 7],
    'Resolution Downsampling': [224, 112, 56, 28, 14, 7],
    'JPEG Compression':        [100, 80, 60, 40, 20, 5],
}

print(f'Device: {DEVICE} | Seed: {SEED}')

## Step 2 — Dataset Download

In [ ]:
if not os.path.exists('data/HAM10000_metadata.csv'):
    from google.colab import files
    print('Upload your kaggle.json:')
    files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
    !unzip -q skin-cancer-mnist-ham10000.zip -d data/
    print('Done!')
else:
    print('Data already present.')

## Step 3 — Load Data

In [ ]:
df = pd.read_csv('data/HAM10000_metadata.csv')
df['label'] = df['dx'].map(CLASS_TO_IDX)

img_dirs = ['data/HAM10000_images_part_1', 'data/HAM10000_images_part_2']
img_path_map = {}
for d in img_dirs:
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith('.jpg'):
                img_path_map[f.replace('.jpg', '')] = os.path.join(d, f)

df['path'] = df['image_id'].map(img_path_map)
df = df.dropna(subset=['path'])

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=SEED)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

# Save test_df to Drive so robustness eval can reload it after disconnect
test_df.to_csv(f'{SAVE_DIR}test_df_seed{SEED}.csv', index=False)
print(f'Test split saved to Drive: test_df_seed{SEED}.csv')

class_counts  = train_df['label'].value_counts().sort_index().values
class_weights = torch.FloatTensor(1.0 / class_counts)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = class_weights.to(DEVICE)

## Step 4 — Degradation Functions

In [ ]:
def apply_gaussian_noise(img_tensor, std):
    if std == 0: return img_tensor
    return torch.clamp(img_tensor + torch.randn_like(img_tensor) * (std / 255.0), 0, 1)

def apply_gaussian_blur(pil_img, radius):
    if radius == 0: return pil_img
    return pil_img.filter(ImageFilter.GaussianBlur(radius=radius))

def apply_downsampling(pil_img, target_size):
    if target_size == IMG_SIZE: return pil_img
    return pil_img.resize((target_size, target_size), Image.BILINEAR).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

def apply_jpeg_compression(pil_img, quality):
    if quality == 100: return pil_img
    buf = io.BytesIO()
    pil_img.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

print('Degradation functions ready.')

## Step 5 — Dataset & DataLoaders

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None, degradation_type=None, severity=0):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.degradation_type = degradation_type
        self.severity = severity

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
        if self.degradation_type == 'Gaussian Blur' and self.severity > 0:
            img = apply_gaussian_blur(img, DEGRADATION_PARAMS['Gaussian Blur'][self.severity])
        elif self.degradation_type == 'Resolution Downsampling' and self.severity > 0:
            img = apply_downsampling(img, DEGRADATION_PARAMS['Resolution Downsampling'][self.severity])
        elif self.degradation_type == 'JPEG Compression' and self.severity > 0:
            img = apply_jpeg_compression(img, DEGRADATION_PARAMS['JPEG Compression'][self.severity])
        if self.transform: img = self.transform(img)
        else: img = transforms.ToTensor()(img)
        if self.degradation_type == 'Gaussian Noise' and self.severity > 0:
            img = apply_gaussian_noise(img, DEGRADATION_PARAMS['Gaussian Noise'][self.severity])
        return img, torch.tensor(row['label'], dtype=torch.long)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_loader = DataLoader(SkinLesionDataset(train_df, transform=train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(SkinLesionDataset(val_df, transform=eval_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Loaders ready.')

## Step 6 — Training

**Each model is saved to Drive immediately after finishing.**  
If Colab disconnects mid-run, already-finished models are safe.  
On reconnect, the notebook detects saved checkpoints and skips retraining them.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += out.argmax(1).eq(labels).sum().item()
        total += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        out = model(imgs)
        total_loss += criterion(out, labels).item() * imgs.size(0)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return (total_loss / len(loader.dataset),
            accuracy_score(all_labels, all_preds),
            f1_score(all_labels, all_preds, average='weighted'))


def train_model(name, model_fn):
    safe_name = name.split()[0].replace("/", "_").replace("-", "_")
    ckpt_path = f'{SAVE_DIR}{safe_name}_seed{SEED}.pth'

    # ── Skip if already trained and saved ──────────────────────
    if os.path.exists(ckpt_path):
        print(f'\nSkipping {name} — checkpoint found at {ckpt_path}')
        model = model_fn().to(DEVICE)
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'Loaded from Drive!')
        return model

    print(f'\n{"="*50}\n Training: {name} | Seed: {SEED}\n{"="*50}')
    model = model_fn().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    best_f1, best_state = 0, None

    for epoch in range(1, EPOCHS + 1):
        train_one_epoch(model, train_loader, optimizer, criterion)
        _, val_acc, val_f1 = evaluate(model, val_loader, criterion)
        scheduler.step()
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = deepcopy(model.state_dict())
        if epoch % 5 == 0:
            print(f'  Epoch {epoch:3d}/{EPOCHS} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

    model.load_state_dict(best_state)
    print(f'  Best Val F1: {best_f1:.4f}')

    # ── Save to Drive immediately ───────────────────────────────
    torch.save(model.state_dict(), ckpt_path)
    print(f'  SAVED TO DRIVE: {ckpt_path}')

    return model


model_builders = {
    'EfficientNetB0 (CNN)':        lambda: timm.create_model('efficientnet_b0',     pretrained=True, num_classes=NUM_CLASSES),
    'ViT-B/16 (Transformer)':      lambda: timm.create_model('vit_base_patch16_224',pretrained=True, num_classes=NUM_CLASSES),
    'EfficientFormer-L1 (Hybrid)': lambda: timm.create_model('efficientformer_l1',  pretrained=True, num_classes=NUM_CLASSES),
}

# ── Train all three — each saved to Drive on completion ─────────
trained_models = {}
for name, builder in model_builders.items():
    trained_models[name] = train_model(name, builder)

print('\nAll models ready!')

## Step 7 — Robustness Evaluation

**If you disconnected after training:** Re-run all cells from the top.  
The training cell will detect Drive checkpoints and reload instantly — no retraining needed.  
Then run this cell.

In [ ]:
@torch.no_grad()
def evaluate_robustness(model, test_df, degradation_type=None, severity=0):
    model.eval()
    dataset = SkinLesionDataset(test_df, transform=eval_transform,
                                degradation_type=degradation_type, severity=severity)
    loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        out = model(imgs.to(DEVICE))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='weighted')


print('Running robustness evaluation (63 conditions)...')
rows = []
for model_name, model in trained_models.items():
    print(f'  Evaluating: {model_name}')
    for deg_type in DEGRADATION_PARAMS:
        for severity in range(6):
            acc, f1 = evaluate_robustness(
                model, test_df,
                degradation_type=deg_type if severity > 0 else None,
                severity=severity
            )
            rows.append({
                'Seed': SEED, 'Model': model_name,
                'Degradation': deg_type, 'Severity': severity,
                'Accuracy': round(acc, 4), 'F1_Weighted': round(f1, 4),
            })

seed_df = pd.DataFrame(rows)

# Save to Drive AND local
local_path = f'robustness_seed{SEED}.csv'
drive_path = f'{SAVE_DIR}robustness_seed{SEED}.csv'
seed_df.to_csv(local_path, index=False)
seed_df.to_csv(drive_path, index=False)

print(f'\nSaved locally: {local_path}')
print(f'Saved to Drive: {drive_path}')
print(seed_df.groupby('Model')[['Accuracy','F1_Weighted']].mean().round(4))

## Step 8 — Download Results

In [ ]:
from google.colab import files
files.download(f'robustness_seed{SEED}.csv')
print(f'Downloaded robustness_seed{SEED}.csv')
print(f'Also permanently saved to Google Drive at: {SAVE_DIR}')
print()
if SEED == 123:
    print('Next step: reopen this notebook, change CURRENT_SEED = 456, run again.')
elif SEED == 456:
    print('Both seeds done! Now run Part 2 (combine all 3 seeds) below.')

---
# PART 2 — Combine All 3 Seeds Into Final Stats

**Run this ONLY after you have all 3 seed CSVs.**

Your files should be in Google Drive at `skin_lesion_research/`:  
- `robustness_seed42.csv` ← rename your original `robustness_results.csv`  
- `robustness_seed123.csv`  
- `robustness_seed456.csv`

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive, files
import os

# Mount Drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/skin_lesion_research/'

# Load all 3 seeds
dfs = []
for seed in [42, 123, 456]:
    # Check Drive first, then local
    drive_path = f'{SAVE_DIR}robustness_seed{seed}.csv'
    local_path = f'robustness_seed{seed}.csv'

    if os.path.exists(drive_path):
        tmp = pd.read_csv(drive_path)
        source = 'Drive'
    elif os.path.exists(local_path):
        tmp = pd.read_csv(local_path)
        source = 'local'
    else:
        print(f'WARNING: robustness_seed{seed}.csv not found — skipping')
        continue

    if 'Seed' not in tmp.columns:
        tmp['Seed'] = seed
    dfs.append(tmp)
    print(f'Loaded seed {seed} from {source}: {len(tmp)} rows')

combined = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(combined)}')
print(f'Seeds: {sorted(combined["Seed"].unique())}')

In [ ]:
# ── Mean ± std across seeds ─────────────────────────────────────
stats = combined.groupby(['Model', 'Degradation', 'Severity']).agg(
    Accuracy_Mean=('Accuracy', 'mean'),
    Accuracy_Std=('Accuracy', 'std'),
    F1_Mean=('F1_Weighted', 'mean'),
    F1_Std=('F1_Weighted', 'std'),
).reset_index().round(4)

stats.to_csv(f'{SAVE_DIR}robustness_stats_final.csv', index=False)
stats.to_csv('robustness_stats_final.csv', index=False)
print('Saved: robustness_stats_final.csv')

# F1 drop summary
print('\nF1 DROP SUMMARY (mean ± std, 3 seeds)')
print('='*72)
drop_rows = []
for model in stats['Model'].unique():
    for deg in stats['Degradation'].unique():
        sub = stats[(stats['Model']==model) & (stats['Degradation']==deg)]
        clean = sub[sub['Severity']==0]['F1_Mean'].values[0]
        worst_row = sub[sub['Severity']==5]
        worst_mean = worst_row['F1_Mean'].values[0]
        worst_std  = worst_row['F1_Std'].values[0]
        drop = (clean - worst_mean) * 100
        print(f'{model[:22]:22s} | {deg[:25]:25s} | {clean:.4f} -> {worst_mean:.4f}±{worst_std:.4f} | drop={drop:.1f}pp')
        drop_rows.append({'Model': model, 'Degradation': deg,
                          'Clean_F1': clean, 'Worst_F1_Mean': worst_mean,
                          'Worst_F1_Std': worst_std, 'F1_Drop_pp': round(drop,1)})

drop_df = pd.DataFrame(drop_rows)
drop_df.to_csv('f1_drop_summary.csv', index=False)
drop_df.to_csv(f'{SAVE_DIR}f1_drop_summary.csv', index=False)
print('\nSaved: f1_drop_summary.csv')

In [ ]:
# ── Robustness curves with error bands ─────────────────────────
import matplotlib.pyplot as plt

MODEL_COLORS  = {'EfficientNetB0 (CNN)': '#e74c3c', 'ViT-B/16 (Transformer)': '#3498db', 'EfficientFormer-L1 (Hybrid)': '#2ecc71'}
MODEL_MARKERS = {'EfficientNetB0 (CNN)': 'o',       'ViT-B/16 (Transformer)': 's',       'EfficientFormer-L1 (Hybrid)': '^'}
DEG_LABELS    = ['Gaussian Noise', 'Gaussian Blur', 'Resolution Downsampling', 'JPEG Compression']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Robustness to Image Degradation — Weighted F1 Score\n(Mean ± Std across 3 random seeds)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, deg_type in enumerate(DEG_LABELS):
    ax = axes[idx]
    for model_name in stats['Model'].unique():
        sub = stats[(stats['Model']==model_name) & (stats['Degradation']==deg_type)].sort_values('Severity')
        sev   = sub['Severity'].values
        means = sub['F1_Mean'].values
        stds  = sub['F1_Std'].values
        color  = MODEL_COLORS.get(model_name, 'gray')
        marker = MODEL_MARKERS.get(model_name, 'o')
        ax.plot(sev, means, color=color, marker=marker, label=model_name, linewidth=2.5, markersize=7)
        ax.fill_between(sev, means-stds, means+stds, alpha=0.15, color=color)
    ax.set_title(deg_type, fontweight='bold', fontsize=12)
    ax.set_xlabel('Severity Level (0=Clean, 5=Worst)')
    ax.set_ylabel('Weighted F1 Score')
    ax.set_xticks(range(6))
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('robustness_curves_final.png', dpi=150, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}robustness_curves_final.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: robustness_curves_final.png')

In [ ]:
# ── Statistical significance: ViT vs CNN ───────────────────────
from scipy import stats as scipy_stats

print('Statistical Significance: ViT vs CNN at worst severity')
print('='*60)
sig_rows = []
for deg_type in DEG_LABELS:
    cnn = combined[(combined['Model']=='EfficientNetB0 (CNN)') &
                   (combined['Degradation']==deg_type) &
                   (combined['Severity']==5)]['F1_Weighted'].values
    vit = combined[(combined['Model']=='ViT-B/16 (Transformer)') &
                   (combined['Degradation']==deg_type) &
                   (combined['Severity']==5)]['F1_Weighted'].values
    if len(cnn) >= 3 and len(vit) >= 3:
        stat, p = scipy_stats.wilcoxon(vit, cnn, alternative='greater')
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'{deg_type:25s} | ViT={np.mean(vit):.4f} CNN={np.mean(cnn):.4f} | p={p:.4f} {sig}')
        sig_rows.append({'Degradation': deg_type, 'ViT_F1': round(np.mean(vit),4),
                         'CNN_F1': round(np.mean(cnn),4), 'p_value': round(p,4), 'significance': sig})

sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv('significance_tests.csv', index=False)
sig_df.to_csv(f'{SAVE_DIR}significance_tests.csv', index=False)
print('\nSaved: significance_tests.csv')
print('* p<0.05  ** p<0.01  *** p<0.001  ns=not significant')

In [ ]:
# ── Download final package ──────────────────────────────────────
import shutil
from google.colab import files

os.makedirs('final_results', exist_ok=True)
for f in ['robustness_stats_final.csv', 'f1_drop_summary.csv',
          'significance_tests.csv', 'robustness_curves_final.png']:
    if os.path.exists(f):
        shutil.copy(f, f'final_results/{f}')

shutil.make_archive('final_results', 'zip', 'final_results')
files.download('final_results.zip')
print('Downloaded final_results.zip')
print(f'Everything also permanently saved to Drive: {SAVE_DIR}')